# Discrete SAC

Read continuous SAC first. Replace sampled continuous-action expectations with exact sums over a finite action set. Train a categorical policy on CartPole.

$$V(s)=\sum_a\pi_\theta(a\mid s)
[\min_i Q_{\bar\phi_i}(s,a)-\alpha\log\pi_\theta(a\mid s)],\qquad
L_\pi=\mathbb E_s\sum_a\pi_\theta(a\mid s)
[\alpha\log\pi_\theta(a\mid s)-\min_i Q_{\phi_i}(s,a)].$$

Here $s,a,r,s'$ denote an observation, action, reward and next observation;
$d$ is one only for true termination; $\gamma$ is the discount factor.
$\theta$ and $\phi_i$ are actor and critic weights; bars denote target weights.
$\mu$ is a deterministic policy, $\pi$ a stochastic policy, and expectations
are minibatch averages from replay. $\alpha$ is the entropy temperature,
$\mathcal H(\pi)=-\mathbb E_a\log\pi(a\mid s)$ is entropy, and $t$ indexes time.
TD3's $\epsilon$ is clipped Gaussian target noise, distinct from behavior noise.
Each implementation below uses only Gymnasium, NumPy and PyTorch.

## 1. Set up the experiment

Use one CPU thread and explicit seeds. The environment determines observation and action dimensions. Hyperparameters are named constants; the default 20,000 steps illustrate learning, not a guaranteed convergence threshold.

In [ ]:
import copy
import math

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

SEED = 7
TOTAL_TIMESTEPS = 20_000
BUFFER_SIZE = 50_000
BATCH_SIZE = 128
LEARNING_STARTS = 1_000
HIDDEN_SIZE = 128
GAMMA = 0.99
TAU = 0.005
ACTOR_LR = 3e-4
CRITIC_LR = 3e-4
MAX_GRAD_NORM = 10.0
EVALUATION_EPISODES = 3
RENDER_MODE = "human"
torch.set_num_threads(1)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

ENV_ID = "CartPole-v1"
probe_env = gym.make(ENV_ID)
try:
    OBS_DIM = int(np.prod(probe_env.observation_space.shape))
    ACTION_DIM = probe_env.action_space.n
    ACTION_START = probe_env.action_space.start
finally:
    probe_env.close()

INITIAL_ALPHA = 0.2
ALPHA_LR = 3e-4
TARGET_ENTROPY = 0.98 * math.log(ACTION_DIM)

## 2. Parameterize categorical probabilities and Q vectors

$$\pi(a\mid s)=\operatorname{softmax}(f_\theta(s))_a.$$

`actor` outputs logits $f_\theta(s)$ and each critic outputs one Q value per
action. `log_softmax` computes log probabilities stably. Behavior samples
from the categorical distribution; deterministic evaluation takes its mode.
No reparameterization or continuous-action Jacobian is needed. Internal action
indices start at zero; `ACTION_START` restores Gymnasium's action offset.

In [ ]:
def mlp(input_dim, output_dim):
    return nn.Sequential(
        nn.Linear(input_dim, HIDDEN_SIZE),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZE, output_dim),
    )


actor = mlp(OBS_DIM, ACTION_DIM)


@torch.no_grad()
def select_action(observation, deterministic=False):
    state = torch.tensor(observation, dtype=torch.float32).reshape(1, OBS_DIM)
    logits = actor(state)
    action = (
        logits.argmax(dim=-1)
        if deterministic
        else torch.distributions.Categorical(logits=logits).sample()
    )
    return int(action.item()) + ACTION_START


critics = nn.ModuleList([mlp(OBS_DIM, ACTION_DIM) for _ in range(2)])
target_critics = copy.deepcopy(critics).requires_grad_(False)
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_optimizer = torch.optim.Adam(critics.parameters(), lr=CRITIC_LR)
log_alpha = torch.tensor(math.log(INITIAL_ALPHA), requires_grad=True)
alpha_optimizer = torch.optim.Adam([log_alpha], lr=ALPHA_LR)

## 3. Store experience and sample replay

$$B=\{(s,a,r,s',d,\text{truncated})\}_1^N\sim\mathcal D.$$

$\mathcal D$ is a circular experience buffer and $N$ is `BATCH_SIZE`.
Uniformly sampled transitions decorrelate consecutive environment steps.
Store the actual behavior action, not the current actor's action. Keep true
termination separate from time-limit truncation: both reset the environment,
but only termination will suppress the bootstrap in the next cell.

In [ ]:
class ReplayBuffer:
    def __init__(self):
        self.observations = np.empty((BUFFER_SIZE, OBS_DIM), dtype=np.float32)
        self.next_observations = np.empty_like(self.observations)
        self.actions = np.empty((BUFFER_SIZE, 1), dtype=np.int64)
        self.rewards = np.empty((BUFFER_SIZE, 1), dtype=np.float32)
        self.terminated = np.empty((BUFFER_SIZE, 1), dtype=np.float32)
        self.truncated = np.empty_like(self.terminated)
        self.position = 0
        self.size = 0

    def add(self, observation, action, reward, next_observation, terminated, truncated):
        i = self.position
        self.observations[i] = np.asarray(observation).reshape(-1)
        self.actions[i] = int(action) - ACTION_START
        self.rewards[i] = reward
        self.next_observations[i] = np.asarray(next_observation).reshape(-1)
        self.terminated[i] = terminated
        self.truncated[i] = truncated
        self.position = (i + 1) % BUFFER_SIZE
        self.size = min(self.size + 1, BUFFER_SIZE)

    def sample(self):
        indices = rng.integers(self.size, size=BATCH_SIZE)
        arrays = (
            self.observations,
            self.actions,
            self.rewards,
            self.next_observations,
            self.terminated,
            self.truncated,
        )
        return tuple(torch.tensor(array[indices]) for array in arrays)


replay = ReplayBuffer()

## 4. Form detached Bellman targets

$$y=r+\gamma(1-d)V_{\mathrm{next}}(s'),\qquad
\bar w\leftarrow(1-\tau)\bar w+\tau w.$$

`terminated` implements $d$; `truncated` is deliberately absent from this mask.
`no_grad` keeps target computation outside the optimization graph.
$w$ denotes online network weights and $\bar w$ their slowly moving target
copies; `TAU` is $\tau$. The soft-update helper only moves network parameters.
$V_{\mathrm{next}}=\sum_a\pi_\theta(a\mid s\prime)[\min_iQ_{\bar\phi_i}(s\prime,a)-\alpha\log\pi_\theta(a\mid s\prime)]$. Sum over all actions instead of sampling one. Only the critics have target copies.

In [ ]:
@torch.no_grad()
def soft_update(online, target):
    for parameter, delayed in zip(
        online.parameters(), target.parameters(), strict=True
    ):
        delayed.lerp_(parameter, TAU)


@torch.no_grad()
def td_target(rewards, next_states, terminated):
    log_probs = actor(next_states).log_softmax(dim=-1)
    q = torch.minimum(target_critics[0](next_states), target_critics[1](next_states))
    value = (log_probs.exp() * (q - log_alpha.exp() * log_probs)).sum(-1, keepdim=True)
    return rewards + GAMMA * (1 - terminated) * value

## 5. Fit the critic and improve the actor

$$L_Q=\sum_i\mathbb E_B[(Q_{\phi_i}(s,a)-y)^2].$$

The critic fits detached targets at recorded behavior actions. The actor then
optimizes its current actions at the sampled states. These are different
uses of the same replay minibatch. Gradient clipping bounds each optimizer's
gradient norm; it does not change the Bellman target.

$$L_\pi=\mathbb E[\alpha\log\pi_\theta(a\mid s)-\min_iQ_{\phi_i}(s,a)],\qquad
L_\beta=-\mathbb E[\beta(\log\pi_\theta(a\mid s)+\mathcal H_*)],\quad \alpha=e^\beta.$$

$\beta$ is `log_alpha` and $\mathcal H_*$ is `TARGET_ENTROPY`.
The log-temperature surrogate increases $\alpha$ when entropy is below target.
Detach the entropy residual so the temperature optimizer cannot change the
actor; detach $\alpha$ in actor and critic updates. The continuous version
estimates action expectations with a reparameterized sample; discrete SAC
sums them exactly. Target critics move after every update.

In [ ]:
def update(batch, update_number):
    states, actions, rewards, next_states, terminated, truncated = batch
    targets = td_target(rewards, next_states, terminated)
    critic_loss = sum(
        nn.functional.mse_loss(critic(states).gather(1, actions), targets)
        for critic in critics
    )

    critic_optimizer.zero_grad(set_to_none=True)
    critic_loss.backward()
    nn.utils.clip_grad_norm_(critics.parameters(), MAX_GRAD_NORM)
    critic_optimizer.step()
    critic_optimizer.zero_grad(set_to_none=True)
    metrics = {"critic_loss": critic_loss.item()}
    log_probs = actor(states).log_softmax(dim=-1)
    probs = log_probs.exp()
    with torch.no_grad():
        q = torch.minimum(critics[0](states), critics[1](states))
    actor_loss = (probs * (log_alpha.detach().exp() * log_probs - q)).sum(-1).mean()
    log_prob = (probs.detach() * log_probs.detach()).sum(-1)

    actor_optimizer.zero_grad(set_to_none=True)
    actor_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
    actor_optimizer.step()
    temperature_loss = -(log_alpha * (log_prob.detach() + TARGET_ENTROPY)).mean()
    alpha_optimizer.zero_grad(set_to_none=True)
    temperature_loss.backward()
    alpha_optimizer.step()
    metrics.update(
        entropy=-log_prob.detach().mean().item(), alpha=log_alpha.detach().exp().item()
    )
    soft_update(critics, target_critics)
    metrics["actor_loss"] = actor_loss.item()
    return metrics

## 6. Interact and learn

During warmup sample uniformly from the action space; afterward use the
exploratory policy. Each new transition enters replay, then one minibatch
update runs once enough experience exists. Reset on `terminated or truncated`
after storing the final observation. The training objective is a discounted
return, while the plotted episode return $R=\sum_t r_{t+1}$ is undiscounted.
Training owns its environment and closes it even if interrupted.

In [ ]:
def train(total_timesteps):
    env = gym.make(ENV_ID)
    episode_returns, critic_losses, actor_losses = [], [], []
    entropies, temperatures = [], []
    episode_return = 0.0
    updates = 0
    try:
        env.action_space.seed(SEED)
        observation, _ = env.reset(seed=SEED)
        for step in range(total_timesteps):
            action = (
                env.action_space.sample()
                if step < LEARNING_STARTS
                else select_action(observation)
            )
            next_observation, reward, terminated, truncated, _ = env.step(action)
            # Store the final observation before resetting at a time limit.
            replay.add(
                observation, action, reward, next_observation, terminated, truncated
            )
            observation = next_observation
            episode_return += float(reward)
            if replay.size >= BATCH_SIZE and step + 1 >= LEARNING_STARTS:
                updates += 1
                metrics = update(replay.sample(), updates)
                critic_losses.append(metrics["critic_loss"])
                if "actor_loss" in metrics:
                    actor_losses.append(metrics["actor_loss"])
                if "entropy" in metrics:
                    entropies.append(metrics["entropy"])
                    temperatures.append(metrics["alpha"])
            if terminated or truncated:
                episode_returns.append(episode_return)
                episode_return = 0.0
                observation, _ = env.reset()
            if (step + 1) % 5_000 == 0:
                recent = np.mean(episode_returns[-10:]) if episode_returns else np.nan
                print(f"Step {step + 1}: recent return={recent:.1f}")
    finally:
        env.close()
    return episode_returns, critic_losses, actor_losses, entropies, temperatures


returns, critic_losses, actor_losses, entropies, temperatures = train(TOTAL_TIMESTEPS)

## 7. Inspect the learning signals

Plot episode returns and their moving mean
$\bar R_k=\frac{1}{W}\sum_{j=k-W+1}^kR_j$, where $W$ is the window size.
Critic loss measures value fitting; actor loss is an optimization objective,
not a return estimate. For SAC also compare entropy with its target and track
$\alpha$. Short runs and different seeds may behave differently.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(returns, alpha=0.3, label="Episode return")
if returns:
    window = min(10, len(returns))
    average = np.convolve(returns, np.ones(window) / window, mode="valid")
    axes[0].plot(np.arange(window - 1, len(returns)), average, label="Moving mean")
axes[0].set(xlabel="Episode", ylabel="Return", title=ENV_ID)
axes[0].legend()
axes[1].plot(critic_losses, label="Critic")
axes[1].set(xlabel="Critic update", ylabel="MSE", title="Value fitting")
axes[2].plot(actor_losses, label="Actor loss")
axes[2].set(xlabel="Actor update", ylabel="Loss", title="Policy optimization")
plt.tight_layout()
plt.show()
if entropies:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(entropies)
    axes[0].axhline(TARGET_ENTROPY, color="black", linestyle="--")
    axes[0].set(xlabel="Update", ylabel="Entropy", title="Entropy and target")
    axes[1].plot(temperatures)
    axes[1].set(xlabel="Update", ylabel="Alpha", title="Learned temperature")
    plt.tight_layout()
    plt.show()

## 8. Evaluate deterministically in a rendered environment

Use a separate environment and new seeds. Deterministic evaluation removes behavior noise or uses the stochastic policy mode; it measures reward without the entropy bonus. The default `human` mode opens a window; set `RENDER_MODE = "rgb_array"` for headless execution.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode=RENDER_MODE)
evaluation_returns = []
try:
    for episode in range(EVALUATION_EPISODES):
        observation, _ = evaluation_env.reset(seed=1_000 + episode)
        total_reward = 0.0
        done = False
        while not done:
            action = select_action(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = evaluation_env.step(action)
            total_reward += float(reward)
            done = terminated or truncated
        evaluation_returns.append(total_reward)
finally:
    evaluation_env.close()
print("Evaluation returns:", evaluation_returns)
print(f"Mean deterministic return: {np.mean(evaluation_returns):.1f}")

## What changed, and what comes next?

Only the action expectation changed: replay, twin critics, target bootstrapping, and temperature tuning remain. Try comparing the categorical policy with DQN on the same environment.

[Original reference](https://arxiv.org/abs/1910.07207) · [Library guide](../../docs/algorithms/discrete_sac.md) · [Public API example](../../examples/discrete_sac.ipynb)